# M1 고정 CLV 구간별 신규상품 오차 진단

기존 Dunnhumby seed 42 M1 체크포인트를 **재학습하지 않고** 분석합니다.

- CLV 구간: 1~683일 학습이력의 고정 `N×V` proxy 기준 저·중·고 구간
- 평가: 684~690일 신규상품 정답만 사용
- 비교 역할: 정답 전체, Top-10 적중, Top-10 누락, 11~20위, 21~50위, 50위 밖, Top-10 오추천
- 상품 비교: 가격 백분위, 학습 구매고객 수, 반복구매 비율, 기존 구매 카테고리와의 일치, 구매이력 임베딩 유사도

이 결과는 고정 CLV 구간별 추천 오차가 실제로 다른지 확인하는 개발용 진단입니다. 모형 선택이나 유의성 주장이 아닙니다.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

REVIEWED_SHA = '6561444bd6a7ba5823c787bb9ee24feb9d7d9540'
%cd /content
!rm -rf /content/clv-m2-lightgcn-runner
!git clone -q https://github.com/jung-un/clv-m2-lightgcn-runner.git /content/clv-m2-lightgcn-runner
%cd /content/clv-m2-lightgcn-runner
!git checkout -q $REVIEWED_SHA

import subprocess
actual_sha = subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip()
assert actual_sha == REVIEWED_SHA, (actual_sha, REVIEWED_SHA)
print('진단 코드 고정:', actual_sha)

In [ ]:
import json
import torch
from lightgcn_clv_fixed_segment_error_diagnostic import (
    configure_fixed_segment_error_diagnostic,
    preflight_summary,
    run_fixed_segment_error_diagnostic,
)

assert torch.cuda.is_available(), '런타임 유형에서 GPU를 선택하세요.'
cfg = configure_fixed_segment_error_diagnostic(
    out_dir=(
        '/content/drive/MyDrive/논문/data/'
        'results_v3_dunnhumby_m1_fixed_clv_segment_error_diagnostic_v1'
    ),
    baseline_result_dir=(
        '/content/drive/MyDrive/논문/data/'
        'results_v3_dunnhumby_m2_repeatshare_historical_backtest_v1'
    ),
    eval_batch_size=32,
    top_examples=20,
)
summary = preflight_summary(cfg)
assert summary['training'] is False
assert summary['checkpoint_selection'] is False
assert summary['split'] == 'historical_development_days_684_690'
assert summary['fixed_clv_source'] == 'train-history N×V proxy at day 683'
print(json.dumps(summary, ensure_ascii=False, indent=2))

In [ ]:
report = run_fixed_segment_error_diagnostic(cfg)

In [ ]:
from IPython.display import display

print('1) 고정 CLV 구간별 M1 성과')
display(report['segment_metrics'])
print('2) 정답·적중·오추천 상품 특성')
display(report['item_role_summary'])
print('3) 정답 누락상품 - Top-10 오추천상품 격차')
display(report['contrasts'])
print('4) 구간·역할별 상위 카테고리')
display(report['category_summary'])
print('5) 실제 정답 누락·오추천 상품 예시')
display(report['examples'])
print('저장 파일:', json.dumps(report['paths'], ensure_ascii=False, indent=2))